# Initial Data Observations

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
departments = pd.read_csv("departments.csv")
diagnosis = pd.read_csv("diagnosis.csv")
encounters = pd.read_csv("encounters.csv")
patients = pd.read_csv("patients.csv")
providers = pd.read_csv("providers.csv")
social_determinants = pd.read_csv("social_determinants.csv")
tiger_census_codes = pd.read_csv("tigercensuscodes.csv")

In [ ]:
departments

In [ ]:
diagnosis

In [ ]:
encounters

In [ ]:
encounters.isnull().sum()

In [ ]:
1531262/7675801

In [ ]:
encounters_null = encounters.loc[encounters["DischargeYear"].isnull()]
encounters_null

In [ ]:
encounters_null.isnull().sum()

In [ ]:
encounters_null["VisitType"].value_counts()

In [ ]:
encounters_not_null = encounters.loc[~encounters["DischargeYear"].isnull()]
encounters_not_null

In [ ]:
encounters_not_null["VisitType"].value_counts()

In [ ]:
encounters_null["Type"].value_counts()

In [ ]:
encounters_not_null["Type"].value_counts()

In [ ]:
departments["Address"].value_counts()

In [ ]:
diagnosis

In [ ]:
encounters_null.loc[encounters_null["VisitType"] == "*Unspecified"]["Type"].value_counts()

In [ ]:
encounters.loc[encounters["IsEdVisit"] == 1]["VisitType"].value_counts()

In [ ]:
encounters_null.loc[encounters["IsEdVisit"] == 1]["VisitType"].value_counts()

In [ ]:
patients

In [ ]:
diagnosis["GroupName"].value_counts()

In [ ]:
encounters_null

In [ ]:
diagnosis.loc[diagnosis["GroupName"] == "Sleep disorders"]

In [ ]:
df = pd.merge(encounters, diagnosis, left_on = "PrimaryDiagnosisKey", right_on = "DiagnosisKey")
df["Depressives"] = df["GroupName"].str.contains("Depressive", case = False, na = False).astype(int)
depressives_df = df.loc[df["Depressives"] == 1]
depressives_df["Date"] = pd.to_datetime(depressives_df["Date"], format = "mixed")

In [ ]:
depressives_df

In [ ]:
# df should be a sub-dataframe of encounters with all encounters involving the relevant diagnoses
# in your input df, make sure you change the "Date" column to datetime objects
def time_between_visits(df):
    results = []
    for patient_key in df["PatientDurableKey"].unique():
        patient_df = df.loc[df["PatientDurableKey"] == patient_key].sort_values(by = "Date", ascending = True)
        
        if len(patient_df) == 1:
            avg_wait = pd.to_timedelta(0, unit = 's')
            
        else: 
            time_btwn = []
            for i in range(0, len(patient_df) - 1):
                time_btwn.append(patient_df.iloc[i+1, 0] - patient_df.iloc[i, 0])
            avg_wait = np.mean(time_btwn)

        results.append({"Patient Key": patient_key, "Avg Wait Time": avg_wait})

    return pd.DataFrame(results)

In [ ]:
depressives_waittime_df = time_between_visits(depressives_df)
depressives_waittime_df

In [ ]:
depressives_waittime_cleaned = depressives_waittime_df.loc[depressives_waittime_df["Avg Wait Time"] > pd.Timedelta(0)]

In [ ]:
depressives_waittime_df.loc[depressives_waittime_df["Avg Wait Time"] > pd.Timedelta(0)].shape[0] / depressives_waittime_df.shape[0]

In [ ]:
sns.histplot(depressives_waittime_cleaned["Avg Wait Time"].dt.days, bins = 30)

In [ ]:
depressives_waittime_cleaned["Avg Wait Time"].describe()

In [ ]:
q1 = depressives_waittime_cleaned["Avg Wait Time"].quantile(0.25)
q3 = depressives_waittime_cleaned["Avg Wait Time"].quantile(0.75)
iqr = q3 - q1
outliers = depressives_waittime_cleaned[(depressives_waittime_cleaned["Avg Wait Time"] < (q1 - 1.5 * iqr)) | (depressives_waittime_cleaned["Avg Wait Time"] > (q3 + 1.5 * iqr))]
outliers

In [ ]:
outliers.describe()

In [ ]:
depressives_df

In [ ]:
county_mapping = pd.DataFrame({
    "CountyCode": [
        "001","003","005","007","009","011","013","015","017","019","021","023","025","027",
        "029","031","033","035","037","039","041","043","045","047","049","051","053","055",
        "057","059","061","063","065","067","069","071","073","075","077","079","081","083",
        "085","087","089","091","093","095","097","099","101","103","105","107","109","111",
        "113","115","117","119","121","123","125","127","129","131","133","135","137","139",
        "141","143","145","147","149","151","153","155","157","159","161","163","165","167",
        "169","171","173","175","177","179","181","183","185","187","189","191","193","195",
        "197","199","201","203","205","207","209"
    ],
    "CountyName": [
        "Allen","Anderson","Atchison","Barber","Barton","Bourbon","Brown","Butler","Chase",
        "Chautauqua","Cherokee","Cheyenne","Clark","Clay","Cloud","Coffey","Comanche","Cowley",
        "Crawford","Decatur","Dickinson","Doniphan","Douglas","Edwards","Elk","Ellis","Ellsworth",
        "Finney","Ford","Franklin","Geary","Gove","Graham","Grant","Gray","Greeley","Greenwood",
        "Hamilton","Harper","Harvey","Haskell","Hodgeman","Jackson","Jefferson","Jewell",
        "Johnson","Kearny","Kingman","Kiowa","Labette","Lane","Leavenworth","Lincoln","Linn",
        "Logan","Lyon","McPherson","Marion","Marshall","Meade","Miami","Mitchell","Montgomery",
        "Morris","Morton","Nemaha","Neosho","Ness","Norton","Osage","Osborne","Ottawa","Pawnee",
        "Phillips","Pottawatomie","Pratt","Rawlins","Reno","Republic","Rice","Riley","Rooks",
        "Rush","Russell","Saline","Scott","Sedgwick","Seward","Shawnee","Sheridan","Sherman",
        "Smith","Stafford","Stanton","Stevens","Sumner","Thomas","Trego","Wabaunsee","Wallace",
        "Washington","Wichita","Wilson","Woodson","Wyandotte"
    ]
})

In [ ]:
patients = patients[
    patients["CensusBlockGroupFipsCode"].notna() &
    (patients["CensusBlockGroupFipsCode"] != "*Unspecified")
] # removing patients with null or unspecified census block group information - unsure whether or not to do this 

tiger_census_codes.head()

tiger_census_codes["GEOID"] = tiger_census_codes["GEOID"].astype(str)
tiger_census_codes["CountyFips"] = tiger_census_codes["GEOID"].str[:5]


tiger_geo = tiger_census_codes[[
    "GEOID",
    "CountyFips",
    "CENTLAT",
    "CENTLON",
    "PopulationValue"
]]


patients_geo = patients.merge(
    tiger_geo,
    left_on="CensusBlockGroupFipsCode",
    right_on="GEOID",
    how="left"
)

patients_geo["CountyCode"] = patients_geo["CountyFips"].str[-3:]

In [ ]:
depressives_county_df = pd.merge(depressives_waittime_cleaned, patients_geo[["DurableKey", "CountyCode"]], left_on = 'Patient Key', right_on = 'DurableKey', how = 'inner')
depressives_county_df

In [ ]:
depressives_county_df = depressives_county_df[["Patient Key", "Avg Wait Time", "CountyCode"]]
depressives_county_df


In [ ]:
depressives_county_wt = (depressives_county_df.groupby("CountyCode").agg(AvgWaitTime = ("Avg Wait Time", "mean"), NumPatients=("Patient Key", "nunique"))
      .reset_index())
depressives_county_wt

In [ ]:
depressives_county_wt_sorted = depressives_county_wt.sort_values("NumPatients", ascending=False)
depressives_county_wt_sorted

In [ ]:
county_mapping.loc[county_mapping["CountyCode"] == "139"]

In [ ]:
county_mapping.loc[county_mapping["CountyCode"] == "111"]

In [ ]:
county_mapping.loc[county_mapping["CountyCode"] == "161"]

In [ ]:
df = pd.merge(encounters, diagnosis, left_on='PrimaryDiagnosisKey', right_on='DiagnosisKey')
df['CANCER'] = df['GroupName'].str.contains('malignant', case=False, na=False).astype(int)
cancer_df = df.loc[df["CANCER"] == 1]
cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format='mixed')

In [ ]:

cancer_waittime_df = time_between_visits(cancer_df)

In [ ]:
cancer_waittime_df = cancer_waittime_df.loc[cancer_waittime_df["Avg Wait Time"] != pd.Timedelta(0, unit = 's')]

In [ ]:
q1 = np.percentile(cancer_waittime_df["Avg Wait Time"], 25)
q3 = np.percentile(cancer_waittime_df["Avg Wait Time"], 75)

IQR = q3 - q1

upper_fence = q3 + 1.5 * IQR
lower_fence = q1 - 1.5 * IQR

outlier_bool = (cancer_waittime_df["Avg Wait Time"] > upper_fence) | (cancer_waittime_df["Avg Wait Time"] < lower_fence)
outliers_cancer = cancer_waittime_df.loc[outlier_bool]

cancer_wt_clean = cancer_waittime_df[~ cancer_waittime_df["Patient Key"].isin(outliers_cancer["Patient Key"])]

In [ ]:
cancer_county_df = pd.merge(cancer_wt_clean, patients_geo[["DurableKey", "CountyCode"]], left_on = 'Patient Key', right_on = 'DurableKey', how = 'inner')
cancer_county_df = cancer_county_df[["Patient Key", "Avg Wait Time", "CountyCode"]]


cancer_county_wt = (cancer_county_df.groupby("CountyCode").agg(AvgWaitTime = ("Avg Wait Time", "mean"), NumPatients=("Patient Key", "nunique"))
      .reset_index())
cancer_county_wt.sort_values("NumPatients", ascending = False)

In [ ]:
departments

In [ ]:
encounters

In [ ]:
patients

In [ ]:
df = pd.merge(df, patients[["DurableKey", "MyChartStatus"]], 
              left_on="PatientDurableKey", right_on="DurableKey", how="left")

In [ ]:
depressives_with_mychart = pd.merge(
    depressives_waittime_df,
    patients[["DurableKey", "MyChartStatus"]],
    left_on="Patient Key",
    right_on="DurableKey",
    how="left"
)

In [ ]:
df = pd.merge(encounters, diagnosis, 
              left_on="PrimaryDiagnosisKey", 
              right_on="DiagnosisKey")

df["FRACTURE"] = df["GroupName"].str.contains("fracture", case=False, na=False).astype(int)

fracture_df = df.loc[df["FRACTURE"] == 1].copy()
fracture_df["Date"] = pd.to_datetime(fracture_df["Date"], format="mixed")

In [ ]:
def time_between_visits(df):
    df = df.sort_values(["PatientDurableKey", "Date"])
    
    df["next_visit"] = df.groupby("PatientDurableKey")["Date"].shift(-1)
    df["gap_days"] = (df["next_visit"] - df["Date"]).dt.days
    
    # remove invalid gaps
    df = df[df["gap_days"] > 0]
    
    result = df.groupby("PatientDurableKey")["gap_days"].mean().reset_index()
    result.columns = ["Patient Key", "Avg Wait Time"]
    
    return result

In [ ]:
fracture_waittime_df = time_between_visits(fracture_df)

In [ ]:
q1 = fracture_waittime_df["Avg Wait Time"].quantile(0.25)
q3 = fracture_waittime_df["Avg Wait Time"].quantile(0.75)

iqr = q3 - q1

fracture_clean = fracture_waittime_df[
    (fracture_waittime_df["Avg Wait Time"] >= q1 - 1.5 * iqr) &
    (fracture_waittime_df["Avg Wait Time"] <= q3 + 1.5 * iqr)
]

In [ ]:
fracture_county_df = pd.merge(
    fracture_clean,
    patients_geo[["DurableKey", "CountyCode"]],
    left_on="Patient Key",
    right_on="DurableKey",
    how="inner"
)

fracture_county_df = fracture_county_df[
    ["Patient Key", "Avg Wait Time", "CountyCode"]
]

In [ ]:
fracture_county_wt = (
    fracture_county_df
    .groupby("CountyCode")
    .agg(
        AvgWaitTime=("Avg Wait Time", "mean"),
        NumPatients=("Patient Key", "nunique")
    )
    .reset_index()
)

fracture_county_wt_sorted = fracture_county_wt.sort_values(
    "NumPatients", ascending=False
)

In [ ]:
fracture_with_mychart = pd.merge(
    fracture_clean,
    patients[["DurableKey", "MyChartStatus"]],
    left_on="Patient Key",
    right_on="DurableKey",
    how="left"
)

In [ ]:
def clean_mychart(x):
    if x == "Activated":
        return "Activated"
    else:
        return "Not Activated"

fracture_with_mychart["MyChartClean"] = fracture_with_mychart["MyChartStatus"].apply(clean_mychart)

In [ ]:
fracture_with_mychart.groupby("MyChartClean")["Avg Wait Time"].mean()

In [ ]:
fracture_geo_mychart = pd.merge(
    fracture_with_mychart,
    patients_geo[["DurableKey", "CountyCode"]],
    left_on="Patient Key",
    right_on="DurableKey",
    how="left"
)

In [ ]:
fracture_mychart_county = (
    fracture_geo_mychart
    .groupby(["CountyCode", "MyChartClean"])
    .agg(
        AvgWaitTime=("Avg Wait Time", "mean"),
        NumPatients=("Patient Key", "count")
    )
    .reset_index()
)

In [ ]:
df["CANCER"] = df["GroupName"].str.contains(
    "malignant|cancer|carcinoma|neoplasm|tumor",
    case=False,
    na=False
).astype(int)

In [ ]:
df = pd.merge(encounters, diagnosis,
              left_on="PrimaryDiagnosisKey",
              right_on="DiagnosisKey")

df = pd.merge(
    df,
    patients[["DurableKey", "MyChartStatus"]],
    left_on="PatientDurableKey",
    right_on="DurableKey",
    how="left"
)

df["CANCER"] = df["GroupName"].str.contains(
    "malignant|cancer|carcinoma|neoplasm|tumor",
    case=False,
    na=False
).astype(int)

cancer_df = df.loc[df["CANCER"] == 1].copy()
cancer_df["Date"] = pd.to_datetime(cancer_df["Date"], format="mixed")

In [ ]:
cancer_waittime_df = time_between_visits(cancer_df)

In [ ]:
q1 = cancer_waittime_df["Avg Wait Time"].quantile(0.25)
q3 = cancer_waittime_df["Avg Wait Time"].quantile(0.75)
iqr = q3 - q1

cancer_clean = cancer_waittime_df[
    (cancer_waittime_df["Avg Wait Time"] >= q1 - 1.5 * iqr) &
    (cancer_waittime_df["Avg Wait Time"] <= q3 + 1.5 * iqr)
]

In [ ]:
cancer_with_mychart = pd.merge(
    cancer_clean,
    patients[["DurableKey", "MyChartStatus"]],
    left_on="Patient Key",
    right_on="DurableKey",
    how="left"
)

In [ ]:
def clean_mychart(x):
    if x == "Activated":
        return "Activated"
    else:
        return "Not Activated"

cancer_with_mychart["MyChartClean"] = cancer_with_mychart["MyChartStatus"].apply(clean_mychart)

In [ ]:
cancer_with_mychart.groupby("MyChartClean")["Avg Wait Time"].mean()

In [ ]:
cancer_with_mychart["LongGap"] = cancer_with_mychart["Avg Wait Time"] > 180

In [ ]:
cancer_with_mychart.groupby("MyChartClean")["LongGap"].mean()

In [ ]:
cancer_with_mychart.groupby("MyChartClean")["Avg Wait Time"].median()

In [ ]:
visit_counts = cancer_df.groupby("PatientDurableKey").size()

cancer_with_mychart["NumVisits"] = cancer_with_mychart["Patient Key"].map(visit_counts)

In [ ]:
cancer_with_mychart.groupby("MyChartClean")["NumVisits"].mean()

In [ ]:
visit_counts = df.groupby("PatientDurableKey").size()
visit_counts

In [ ]:
journey_span = df.groupby("PatientDurableKey")["Date"].agg(
    lambda x: (x.max() - x.min()).days
)

In [ ]:
df.columns

In [ ]:
df = df.copy()

df["Date"] = pd.to_datetime(df["Date"], format="mixed")

# drop rows without patient or date
df = df.dropna(subset=["PatientDurableKey", "Date"])

In [ ]:
visit_counts = df.groupby("PatientDurableKey").size()

In [ ]:
dept_counts = df.groupby("PatientDurableKey")["DepartmentKey"].nunique()

In [ ]:
journey_duration = df.groupby("PatientDurableKey")["Date"].agg(
    lambda x: (x.max() - x.min()).days
)

In [ ]:
inpatient_usage = df.groupby("PatientDurableKey")["IsInpatientAdmission"].sum()

In [ ]:
ed_usage = df.groupby("PatientDurableKey")["IsEdVisit"].sum()

In [ ]:
journey_df = pd.DataFrame({
    "PatientKey": visit_counts.index,
    "NumVisits": visit_counts.values,
    "NumDepartments": dept_counts.values,
    "Duration": journey_duration.values,
    "EDVisits": ed_usage.values,
    "InpatientVisits": inpatient_usage.values
})

In [ ]:
journey_df

In [ ]:
journey_df["LongDuration"] = journey_df["Duration"] > 180
journey_df["HighUtilization"] = journey_df["NumVisits"] > 5
journey_df["Fragmented"] = journey_df["NumDepartments"] > 3

In [ ]:
journey_df[["LongDuration", "HighUtilization", "Fragmented"]].mean()

In [ ]:
journey_df["ComplexityScore"] = (
    journey_df["NumVisits"] * 0.5 +
    journey_df["NumDepartments"] * 0.3 +
    journey_df["Duration"] * 0.2
)

In [ ]:
journey_geo = pd.merge(
    journey_df,
    patients_geo[["DurableKey", "CountyCode"]],
    left_on="PatientKey",
    right_on="DurableKey",
    how="left"
)

In [ ]:
county_complexity = (
    journey_geo
    .groupby("CountyCode")
    .agg(
        AvgComplexity=("ComplexityScore", "mean"),
        NumPatients=("PatientKey", "count")
    )
    .reset_index()
)

In [ ]:
from sklearn.cluster import KMeans

features = journey_df[["NumVisits", "NumDepartments", "Duration"]]

kmeans = KMeans(n_clusters=4, random_state=0)
journey_df["Cluster"] = kmeans.fit_predict(features)

In [ ]:
cluster_df = journey_df.groupby("Cluster").mean()
cluster_df

In [ ]:
df_cancer = df[df["CANCER"] == 1]

In [ ]:
import matplotlib.pyplot as plt

cluster_summary = journey_df.groupby("Cluster").mean()

cluster_summary[["NumVisits", "NumDepartments", "Duration"]].plot(kind="bar")
plt.title("Patient Journey Profiles by Cluster")
plt.ylabel("Average Value")
plt.xticks(rotation=0)
plt.show()

In [ ]:
journey_df["Cluster"].value_counts(normalize=True).sort_index().plot(kind="bar")

plt.title("Distribution of Patient Types")
plt.ylabel("Proportion of Patients")
plt.xlabel("Cluster")
plt.show()

In [ ]:
plt.scatter(journey_df["NumDepartments"], journey_df["Duration"], alpha=0.3)
plt.xlabel("Number of Departments")
plt.ylabel("Duration (days)")
plt.title("Fragmentation vs Care Duration")
plt.show()

In [ ]:
import matplotlib.pyplot as plt

for c in journey_df["Cluster"].unique():
    subset = journey_df[journey_df["Cluster"] == c]
    plt.scatter(subset["NumVisits"], subset["Duration"], label=f"Cluster {c}", alpha=0.4)

plt.xlabel("Number of Visits")
plt.ylabel("Duration (days)")
plt.title("Patient Journey Clusters")
plt.legend()
plt.show()

In [ ]:
county_complexity.sort_values("AvgComplexity", ascending=False)

In [ ]:
county_complexity

In [ ]:
county_complexity_filtered = county_complexity.loc[(county_complexity["CountyCode"] != "177") & (county_complexity["NumPatients"] > 400)]

In [ ]:
county_complexity_filtered

In [ ]:
# visualize avgcomplexity and numpatients and get rid of county 177

plt.figure(figsize=(12, 6))
plt.scatter(county_complexity_filtered["AvgComplexity"], county_complexity_filtered["NumPatients"], alpha=0.6)
plt.title("Average Complexity vs Number of Patients")
plt.xlabel("Average Complexity")
plt.ylabel("Number of Patients")
plt.grid()
plt.show()